# 06 - Build the modelling dataset

Joins, for one site, the daily ETa from the tower (target), the merged vegetation indices
(interpolated to daily) and the CoAgMet weather data. The result is the table used by the ML notebooks.

The end of the notebook has a short look at precipitation over the study period.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

In [ ]:
SITE = "ASP"                  # "ASP" or "BAU"
VI_SOURCE = "fixed_footprint" # which index tables to use (see notebook 05)

ETA_CSV = Path(f"../data/processed/ec/{SITE}_daily_eta.csv")
VI_CSV = Path(f"../data/processed/vi_tables/{VI_SOURCE}/merged_{SITE}.csv")
CLIMATE_CSV = Path("../data/raw/climate/CoAgMet_Climatic_Data.csv")
OUT_DIR = Path("../data/model_input")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
eta = pd.read_csv(ETA_CSV)
vis = pd.read_csv(VI_CSV)
climate = pd.read_csv(CLIMATE_CSV)

eta = eta.rename(columns={"ET_mm_day": "ETa"})   # older files used this name
eta["date"] = pd.to_datetime(eta["date"])
vis["Date"] = pd.to_datetime(vis["Date"])
climate["Date"] = pd.to_datetime(climate["Date"])

## Daily vegetation indices

HLS gives a clear observation every few days at best, so the index series are linearly
interpolated to a daily step before joining with the tower data.

In [ ]:
vis = vis.set_index("Date").sort_index()
vis = vis.reindex(pd.date_range(vis.index.min(), vis.index.max(), freq="D"))
vis = vis.interpolate(method="linear")
vis = vis.reset_index().rename(columns={"index": "Date"})

# 'EVI_median' -> 'EVI Median', the names used in the modelling notebooks
vis.columns = [c.replace("_mean", " Mean").replace("_median", " Median") for c in vis.columns]

## Join on the ETa dates

ETa is the reference: every day with a tower value is kept, and indices and weather are attached to it.

In [ ]:
dataset = (
    eta.merge(vis, left_on="date", right_on="Date", how="left")
       .merge(climate, left_on="date", right_on="Date", how="left")
       .drop(columns=["Date_x", "Date_y"], errors="ignore")
)

out_path = OUT_DIR / f"{SITE}_model_input.csv"
dataset.to_csv(out_path, index=False)
print(f"{len(dataset)} days, {dataset.shape[1]} columns -> {out_path}")
dataset.head()

## Precipitation over the study period

Daily precipitation from the CoAgMet station, with the October to mid-April window shaded.
The bar chart on the right is the precipitation accumulated in each of those windows,
which sets the soil water available at the start of the growing season.

In [ ]:
years = list(range(2019, 2025))

def window_total(yr):
    start, end = pd.Timestamp(f"{yr - 1}-10-01"), pd.Timestamp(f"{yr}-04-15")
    sel = climate[(climate["Date"] >= start) & (climate["Date"] <= end)]
    return sel["Precipitation"].sum()

accum_precip = [window_total(y) for y in years]
print(dict(zip(years, [round(v, 1) for v in accum_precip])))

In [ ]:
precip = climate[(climate["Date"] >= "2018-10-01") & (climate["Date"] <= "2024-12-31")]

fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[4.8, 1.3], wspace=0.18)

ax1 = fig.add_subplot(gs[0])
ax1.bar(precip["Date"], precip["Precipitation"], width=1, color="royalblue")
for yr in years:
    ax1.axvspan(pd.Timestamp(f"{yr - 1}-10-01"), pd.Timestamp(f"{yr}-04-15"),
                color="lightgray", alpha=0.45, zorder=0)
ax1.set_xlim(pd.Timestamp("2018-10-01"), pd.Timestamp("2024-12-31"))
ax1.set_ylabel("Daily precipitation (mm)", fontsize=12)
ax1.set_title("Daily precipitation", fontsize=15)
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.legend(handles=[Patch(facecolor="lightgray", edgecolor="gray", alpha=0.45, label="Oct - mid Apr")],
           loc="upper left")

ax2 = fig.add_subplot(gs[1])
bars = ax2.bar(range(len(years)), accum_precip, width=0.65, color="steelblue")
ax2.set_xticks(range(len(years)))
ax2.set_xticklabels(years, fontsize=11)
ax2.set_ylabel("Accumulated P (mm)", fontsize=11)
ax2.set_title("Oct - mid Apr\naccumulated P", fontsize=14)
ax2.set_ylim(0, max(accum_precip) * 1.25)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
for bar, val in zip(bars, accum_precip):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 1, f"{val:.1f}",
             ha="center", fontsize=11, fontweight="bold")

fig.savefig("../results/figures/precipitation_overview.png", dpi=300, bbox_inches="tight")
plt.show()